In [1]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="experiment_1a",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=True,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

In [2]:
import os
import json
import pandas as pd
from datasets import load_dataset
from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

MODEL = "gpt-4.1-mini"   # cheap + good for structured classification prototypes

LABELS = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}
ALLOWED_LABEL_NAMES = list(LABELS.values())


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
# AG News has text classification examples and 4 labels.
dataset = load_dataset("ag_news", split="test")

# Keep only 10 random rows for a lightweight prototype.
RANDOM_SEED = 42
small_ds = dataset.shuffle(seed=RANDOM_SEED).select(range(10))
df = pd.DataFrame(small_ds)
vault.create_record_list("ag_news_small", column_names=["text", "label"])
for i, row in df.iterrows():
    vault.append_record("ag_news_small", 
                        {
                            "text": row["text"],
                            "label": row["label"],
                        }
                       )

description = "This list stores a 10-row random sample from the AG News test split used as the input corpus for experiment_1a. Each record contains raw news article text and the original integer AG News label, where 0=World, 1=Sports, 2=Business, and 3=Sci/Tech. Use this list to inspect source articles, recover ground-truth categories, or trace downstream classification outputs back to the original examples."
embedding = get_embeddings(description)
vault.create_description("ag_news_small", description, embedding)


---[ TableVault Record ]---


---[ TableVault Record ]---



In [10]:
SYSTEM_PROMPT = """You are a careful text classification assistant.

Classify each news article into exactly one of these labels:
- World
- Sports
- Business
- Sci/Tech

Return strict JSON with keys:
- predicted_label
- short_reason

The predicted_label must be exactly one of:
["World", "Sports", "Business", "Sci/Tech"]

Do not use markdown code fences.
"""

def extract_first_json_object(raw_text: str) -> dict:
    if not raw_text or not raw_text.strip():
        raise ValueError("Model returned empty output.")

    decoder = json.JSONDecoder()
    for i, ch in enumerate(raw_text):
        if ch != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(raw_text[i:])
        except json.JSONDecodeError:
            continue
        if isinstance(candidate, dict):
            return candidate

    raise ValueError(f"Could not find JSON object in model output: {raw_text!r}")


def classify_text(text: str) -> dict:
    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Article text:\n\n{text}\n\nClassify this article."
            },
        ],
    )

    raw_text = response.output_text or ""
    result = extract_first_json_object(raw_text)

    if result.get("predicted_label") not in ALLOWED_LABEL_NAMES:
        raise ValueError(f"Unexpected label: {result.get('predicted_label')}")

    if not isinstance(result.get("short_reason"), str):
        raise ValueError("Missing or invalid 'short_reason' in model output.")

    return {
        "predicted_label": result["predicted_label"],
        "short_reason": result["short_reason"],
    }


---[ TableVault Record ]---
---[ TableVault Record ]---



In [11]:
vault.create_record_list("base_output", column_names=["gold_label", "predicted_label", "short_reason"])
description = "This list stores per-article classification outputs for experiment_1a generated from ag_news_small with gpt-4.1-mini. Each record contains the gold_label mapped to AG News label names, the model predicted_label, and a short_reason explaining the prediction. Records are linked to the corresponding source article in ag_news_small and are intended for error analysis, prediction auditing, label comparison, and evaluation of zero-shot news topic classification."
embedding = get_embeddings(description)
vault.create_description("base_output", description, embedding)

for i, row in df.iterrows():
    result = classify_text(row["text"])
    input_items = {"ag_news_small": [i, i + 1]}
    vault.append_record("base_output",
                        {
                            "gold_label": LABELS[row["label"]],
                            "predicted_label": result["predicted_label"],
                            "short_reason": result["short_reason"],
                        }
                        , input_items = input_items
                       )




---[ TableVault Record ]---


DuplicateItemError: Item 'base_output' already exists in 'items' as 'record_list'. Cannot create as 'record_list'.

---[ TableVault Record ]---



In [7]:
results_df = pd.DataFrame(vault.query_item_content("base_output"))
results_df


---[ TableVault Record ]---


,gold_label,predicted_label,short_reason
0,Sports,Sports,The article discusses the Indian cricket board...
1,Business,Business,The article discusses stock market movements a...
2,Sports,Sports,The article reports on a basketball game with ...
3,Business,Business,The article discusses stock market movements a...
4,Sci/Tech,Sci/Tech,The article discusses a video game and its tec...
5,World,Business,The article focuses on China's inflation rate ...
6,Business,Business,The article promotes currency trading services...
7,Business,Business,Article discusses oil production capacity and ...
8,Sci/Tech,Sci/Tech,The article discusses Intel's technology devel...
9,Business,Business,The article discusses a major acquisition in t...


---[ TableVault Record ]---



In [8]:
results_df["correct"] = results_df["gold_label"] == results_df["predicted_label"]
accuracy = results_df["correct"].mean()
vault.create_record_list("experiment_1a_accuracy", column_names=["accuracy"])
description = "This list stores the aggregate exact-match accuracy for experiment_1a over the 10 sampled AG News test articles. The single accuracy value is computed from base_output as the mean of gold_label == predicted_label, summarizing overall classification performance for the run rather than per-record details. Use it for experiment-level evaluation, metric tracking, and comparison with future prompts or models."
embedding = get_embeddings(description)
vault.create_description("experiment_1a_accuracy", description, embedding)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [9]:
vault.append_record("experiment_1a_accuracy", {"accuracy": accuracy})
accuracy


---[ TableVault Record ]---


np.float64(0.9)

---[ TableVault Record ]---

